
# Microsoft Foundry Local 
## Advanced Function Calling with Foundry Local

<img src="https://github.com/retkowsky/foundry-local/blob/main/foundrylocal.jpg?raw=true">

This notebook demonstrates **end-to-end function calling** with Foundry Local, including:

- Defining a rich set of tools (weather, stock prices, unit conversion, web search, email)
- Parsing the model's tool-call responses
- Executing mock tool functions locally
- Feeding results back to the model for a final answer
- Handling **multiple** function calls followed by a final answer

> **Docs:** [Foundry Local Documentation](https://learn.microsoft.com/azure/foundry-local/)  
> **SDK Reference:** [Python SDK Reference](https://learn.microsoft.com/azure/foundry-local/reference/reference-sdk-current?pivots=programming-language-python)

## Author

| Field | Details |
| --- | --- |
| Name | Serge Retkowsky |
| Email | serge.retkowsky@microsoft.com |
| LinkedIn | https://www.linkedin.com/in/serger/ |
| Medium publications | https://medium.com/@sergems18/ |

## Current Foundry Local SDK setup

These examples use the **1.2.4 Python SDK** (`foundry_local_sdk`), not the legacy
`foundry_local` package. Use Python 3.11+ (validated with Python 3.12).
Uncomment the installation command for your OS, run it once, then restart the kernel.
Do not install both SDK distributions in the same environment.

The SDK hosts its own runtime; installing or upgrading the CLI does not upgrade the
Python SDK. Catalog entries are models with hardware-specific `variants`; metadata
is available through `model.info`, with device/provider under `info.runtime`.
Aliases are case-sensitive. `model.download()` downloads the selected variant;
`model.load()` loads it into this SDK runtime.

By default, the SDK uses an application-specific cache. To reuse an existing cache,
set `FOUNDRY_MODEL_CACHE_DIR` before starting the kernel (the CLI shows its cache with
`foundry cache location`). An optional `FOUNDRY_MODEL_ALIAS` selects a different
model for notebooks 01-05; notebooks 02-03 require system-role support, notebook 04 requires Mistral's user/assistant prompt format,
and notebook 05 requires a model that supports tools.

CPU inference works without extra provider downloads. For supported GPU/NPU
acceleration, run `manager.download_and_register_eps()` before querying the catalog.
Provider support differs between SDK and CLI builds: a cached WebGPU model is not
necessarily loadable by the Windows Python SDK. Select/download a supported CPU
variant if necessary. Model and provider downloads require internet access; inference
is local after setup.

References: [SDK](https://learn.microsoft.com/azure/foundry-local/reference/reference-sdk-current?pivots=programming-language-python),
[OpenAI integration](https://learn.microsoft.com/azure/foundry-local/how-to/how-to-integrate-with-inference-sdks).


In [1]:
import datetime
import GPUtil
import json
import openai
import os
import pandas as pd
import platform
import psutil
import random
import sys

from datetime import datetime, timedelta
from foundry_local_sdk import Configuration, FoundryLocalManager

In [2]:
print(f"Python version: {sys.version}")

Python version: 3.12.12 | packaged by Anaconda, Inc. | (main, Oct 21 2025, 20:05:38) [MSC v.1929 64 bit (AMD64)]


In [3]:
print(f"Today is {datetime.today().strftime('%d-%b-%Y %H:%M:%S')}")

Today is 02-Nov-2026 10:46:08


In [4]:
print(f"💻 OS: {platform.system()} {platform.release()}")
print(f"- CPU: {platform.processor()}")
print(
    f"- CPU cores: {psutil.cpu_count(logical=False)} physical, {psutil.cpu_count()} logical"
)

ram = psutil.virtual_memory()
print(f"- RAM total:     {ram.total / (1024**3):.1f} GB")
print(f"- RAM available: {ram.available / (1024**3):.1f} GB")
print(f"- RAM used:      {ram.percent}%")

for part in psutil.disk_partitions():
    try:
        usage = psutil.disk_usage(part.mountpoint)
        print(f"\n💾 Disk [{part.device}] mounted on {part.mountpoint}")
        print(f"- Total: {usage.total / (1024**3):.1f} GB")
        print(f"- Used:  {usage.used / (1024**3):.1f} GB ({usage.percent}%)")
        print(f"- Free:  {usage.free / (1024**3):.1f} GB")
    except PermissionError:
        pass

gpus = GPUtil.getGPUs()

if not gpus:
    print("No GPU detected.")
else:
    for i, gpu in enumerate(gpus):
        print(f"\n🎮 GPU {i} — {gpu.name}")
        print(f"- VRAM Total : {gpu.memoryTotal:,.0f} MB")
        print(
            f"- VRAM Used  : {gpu.memoryUsed:,.0f} MB ({gpu.memoryUtil * 100:.0f}%)"
        )
        print(f"- VRAM Free  : {gpu.memoryFree:,.0f} MB")
        print(f"- GPU Load   : {gpu.load * 100:.0f}%")
        print(f"- Temperature: {gpu.temperature} °C")

💻 OS: Windows 11
- CPU: Intel64 Family 6 Model 141 Stepping 1, GenuineIntel
- CPU cores: 8 physical, 16 logical
- RAM total:     63.7 GB
- RAM available: 18.2 GB
- RAM used:      71.5%

💾 Disk [C:\] mounted on C:\
- Total: 951.6 GB
- Used:  918.3 GB (96.5%)
- Free:  33.4 GB

🎮 GPU 0 — NVIDIA T1200 Laptop GPU
- VRAM Total : 4,096 MB
- VRAM Used  : 3,784 MB (92%)
- VRAM Free  : 152 MB
- GPU Load   : 56%
- Temperature: 68.0 °C


In [5]:
if FoundryLocalManager.instance is None:
    FoundryLocalManager.initialize(Configuration(
        app_name="foundry-local-notebooks",
        model_cache_dir=os.getenv("FOUNDRY_MODEL_CACHE_DIR"),
    ))
manager = FoundryLocalManager.instance
if not manager.urls:
    manager.start_web_service()
endpoint = manager.urls[0].rstrip("/") + "/v1"
print(f"Endpoint (v1): {endpoint}")
print("Cached model paths:", [m.get_path() for m in manager.catalog.get_cached_models()])

Endpoint (v1): http://127.0.0.1:62072/v1
Cached model paths: ['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4', 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']


In [6]:
catalog = [v.info for m in manager.catalog.list_models() for v in m.variants]
print(f"Total model in catalog = {len(catalog)}")

Total model in catalog = 56


In [7]:
for idx, model in enumerate(catalog, start=1):
    print(f"{idx}: {model.id} - {model.runtime.device_type}")

1: deepseek-r1-distill-qwen-1.5b-generic-cpu:4 - CPU
2: deepseek-r1-distill-qwen-14b-generic-cpu:4 - CPU
3: deepseek-r1-distill-qwen-7b-generic-cpu:4 - CPU
4: deepseek-r1-distill-llama-8b-generic-cpu:4 - CPU
5: gemma-4-e2b-it-generic-cpu:3 - CPU
6: gpt-oss-20b-generic-cpu:1 - CPU
7: ministral-3-3b-instruct-2512-generic-cpu:2 - CPU
8: mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:3 - CPU
9: mistral-nemo-12b-instruct-generic-cpu:1 - CPU
10: nemotron-3.5-asr-streaming-0.6b-generic-cpu:3 - CPU
11: nemotron-speech-streaming-en-0.6b-generic-cpu:3 - CPU
12: nemotron-speech-streaming-es-0.6b-ft-generic-cpu:1 - CPU
13: olmo-3-7b-instruct-generic-cpu:1 - CPU
14: parakeet-tdt-0.6b-v2-generic-cpu:1 - CPU
15: Phi-3-mini-128k-instruct-generic-cpu:3 - CPU
16: Phi-3-mini-4k-instruct-generic-cpu:3 - CPU
17: Phi-3.5-mini-instruct-generic-cpu:2 - CPU
18: Phi-4-generic-cpu:2 - CPU
19: Phi-4-mini-instruct-generic-cpu:5 - CPU
20: Phi-4-mini-instruct-generic-cpu-dq44:1 - CPU
21: Phi-4-mini-reasoning-generic

In [8]:
def models_to_df(models):
    """Convert a list of FoundryModelInfo objects into a clean DataFrame."""
    return pd.DataFrame([{
        "alias": m.alias,
        "id": m.id,
        "device": (m.runtime.device_type.value if m.runtime else "unknown"),
        "provider": (m.runtime.execution_provider if m.runtime else "unknown"),
        "size_mb": m.file_size_mb,
        "tools": m.supports_tool_calling,
        "license": m.license,
        "task": m.task,
    } for m in models])

In [9]:
catalog = [v.info for m in manager.catalog.list_models() for v in m.variants]
print(f"Total model variants in catalog = {len(catalog)}")

df_catalog = models_to_df(catalog)
df_catalog

Total model variants in catalog = 56


,alias,id,device,provider,size_mb,tools,license,task
0,deepseek-r1-1.5b,deepseek-r1-distill-qwen-1.5b-generic-cpu:4,CPU,CPUExecutionProvider,1966,None,MIT,chat-completion
1,deepseek-r1-14b,deepseek-r1-distill-qwen-14b-generic-cpu:4,CPU,CPUExecutionProvider,11786,None,MIT,chat-completion
2,deepseek-r1-7b,deepseek-r1-distill-qwen-7b-generic-cpu:4,CPU,CPUExecutionProvider,6584,None,MIT,chat-completion
3,deepseek-r1-8b,deepseek-r1-distill-llama-8b-generic-cpu:4,CPU,CPUExecutionProvider,6796,None,MIT,chat-completion
4,gemma-4-e2b-it,gemma-4-e2b-it-generic-cpu:3,CPU,CPUExecutionProvider,6031,True,apache-2.0,vision-language-chat
5,gpt-oss-20b,gpt-oss-20b-generic-cpu:1,CPU,CPUExecutionProvider,12552,None,MIT,chat-completion
6,ministral-3-3b-instruct-2512,ministral-3-3b-instruct-2512-generic-cpu:2,CPU,CPUExecutionProvider,4691,True,apache-2.0,vision-language-chat
7,mistral-7b-v0.2,mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:3,CPU,CPUExecutionProvider,4167,None,apache-2.0,chat-completion
8,mistral-nemo-12b-instruct,mistral-nemo-12b-instruct-generic-cpu:1,CPU,CPUExecutionProvider,6978,True,apache-2.0,chat-completion
9,nemotron-3.5-asr-streaming-0.6b,nemotron-3.5-asr-streaming-0.6b-generic-cpu:3,CPU,CPUExecutionProvider,756,None,MIT,automatic-speech-recognition


## Choose a tool-capable model

The default is `qwen3-1.7b`, which advertises tool calling. `phi-4` is not a suitable
default: the current catalog does not advertise tool calling for it. You can select
another tool-capable alias with `FOUNDRY_MODEL_ALIAS`, for example `phi-4-mini`.
All tool implementations below are simulations; they do not send email or fetch live data.

In [10]:
MODEL_ALIAS = os.getenv("FOUNDRY_MODEL_ALIAS", "qwen3-1.7b")

In [11]:
MODEL_ALIAS = os.getenv("FOUNDRY_MODEL_ALIAS", "qwen3-1.7b")
model = manager.catalog.get_model(MODEL_ALIAS)
if model is None:
    raise ValueError(f"Model alias {MODEL_ALIAS!r} was not found in the catalog.")
if not model.supports_tool_calling:
    raise ValueError(f"Model {MODEL_ALIAS!r} does not advertise tool calling.")
print(f"Selected: {model.id} ({model.info.runtime.execution_provider})")
if not model.is_cached:
    model.download()

Selected: qwen3-1.7b-generic-cpu:2 (CPUExecutionProvider)


In [12]:
[m.get_path() for m in manager.catalog.get_cached_models()]

['C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\gpt-oss-20b-generic-cpu-1\\v1',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\mistralai-Mistral-7B-Instruct-v0-2-generic-cpu-3\\v3',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\Phi-4-mini-instruct-generic-cpu-5\\v5',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen2.5-0.5b-instruct-generic-cpu-4\\v4',
 'C:\\Users\\seretkow/.foundry-local-notebooks/cache/models\\Microsoft\\qwen3-1.7b-generic-cpu-2\\v2']

In [13]:
cached = [m.info for m in manager.catalog.get_cached_models()]
print(f"Cached models = {len(cached)}")

print(cached)

Cached models = 5
[ModelInfo(id='gpt-oss-20b-generic-cpu:1', name='gpt-oss-20b-generic-cpu', version=1, alias='gpt-oss-20b', display_name='gpt-oss-20b-generic-cpu', provider_type='FoundryLocal', uri='azureml://registries/azureml/models/gpt-oss-20b-generic-cpu/versions/1', model_type='ONNX', prompt_template=None, publisher='Microsoft', model_settings=None, license='MIT', license_description='This model is provided under the License Terms available at <https://huggingface.co/openai/gpt-oss-20b/blob/main/LICENSE>.', task='chat-completion', runtime=Runtime(device_type=<DeviceType.CPU: 'CPU'>, execution_provider='CPUExecutionProvider'), file_size_mb=12552, supports_tool_calling=None, max_output_tokens=2048, min_fl_version='0.0.0', created_at_unix=1764808747, context_length=None, input_modalities='text', output_modalities='text', capabilities=None), ModelInfo(id='mistralai-Mistral-7B-Instruct-v0-2-generic-cpu:3', name='mistralai-Mistral-7B-Instruct-v0-2-generic-cpu', version=3, alias='mistra

In [14]:
model.load()
model_info = model.info
print(f"Loaded: {model_info.alias} => {model_info.id}")

model_id = model.id

Loaded: qwen3-1.7b => qwen3-1.7b-generic-cpu:2


In [15]:
info = model.info
print(f"Running on: {info.runtime.execution_provider}")
print(f"Device:     {info.runtime.device_type.value}")

Running on: CPUExecutionProvider
Device:     CPU


In [16]:
client = openai.OpenAI(
    base_url=endpoint,
    api_key="not-needed",
)

print(f"Connected model: {model_id}")
print(f"endpoint: {endpoint}")

Connected model: qwen3-1.7b-generic-cpu:2
endpoint: http://127.0.0.1:62072/v1


## Define the Tools catalogue

In [17]:
# OpenAI-compatible tool definitions for the API
TOOLS_OPENAI = [{
    "type": "function",
    "function": {
        "name": "get_weather",
        "description": "Get current weather and 3-day forecast for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "City name, e.g. 'Paris' or 'Tokyo'"
                },
                "units": {
                    "type": "string",
                    "description":
                    "Temperature unit: 'celsius' or 'fahrenheit'",
                    "enum": ["celsius", "fahrenheit"]
                }
            },
            "required": ["city"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "get_stock_price",
        "description":
        "Get the latest stock price, daily change, and 52-week range for a ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "ticker": {
                    "type": "string",
                    "description": "Stock ticker symbol, e.g. 'AAPL', 'MSFT'"
                }
            },
            "required": ["ticker"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description":
        "Convert an amount from one currency to another using current exchange rates.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {
                    "type": "number",
                    "description": "The amount of money to convert"
                },
                "from_currency": {
                    "type": "string",
                    "description": "ISO 4217 code, e.g. 'USD'"
                },
                "to_currency": {
                    "type": "string",
                    "description": "ISO 4217 code, e.g. 'EUR'"
                }
            },
            "required": ["amount", "from_currency", "to_currency"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "search_web",
        "description":
        "Search the web and return top results with title, snippet, and URL.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query string"
                },
                "max_results": {
                    "type": "integer",
                    "description": "Number of results (1-10)"
                }
            },
            "required": ["query"]
        }
    }
}, {
    "type": "function",
    "function": {
        "name": "send_email",
        "description": "Send an email to a recipient with subject and body.",
        "parameters": {
            "type": "object",
            "properties": {
                "to": {
                    "type": "string",
                    "description": "Recipient email address"
                },
                "subject": {
                    "type": "string",
                    "description": "Email subject line"
                },
                "body": {
                    "type": "string",
                    "description": "Email body text"
                }
            },
            "required": ["to", "subject", "body"]
        }
    }
}]

# Simple tool list for the system prompt (model-friendly format)
TOOLS_SIMPLE = [{
    "name": t["function"]["name"],
    "description": t["function"]["description"],
    "parameters": t["function"]["parameters"]["properties"]
} for t in TOOLS_OPENAI]

print(f"Registered {len(TOOLS_OPENAI)} tools:")

for tool in TOOLS_OPENAI:
    f = tool['function']
    print(f"- {f['name']}: {f['description']}")

Registered 5 tools:
- get_weather: Get current weather and 3-day forecast for a city.
- get_stock_price: Get the latest stock price, daily change, and 52-week range for a ticker symbol.
- convert_currency: Convert an amount from one currency to another using current exchange rates.
- search_web: Search the web and return top results with title, snippet, and URL.
- send_email: Send an email to a recipient with subject and body.


## System prompt and tool schemas

Pass the function schemas through the API's `tools` parameter. The runtime returns
structured `message.tool_calls`; do not ask the model to invent a `functools[...]`
text protocol. Reasoning models such as Qwen3 can take longer and need a larger
output-token budget than non-reasoning models such as phi-4-mini.

Do not force `/no_think` with this tested CLI build: it can place the answer in
`reasoning_content` and leave `content` empty.


In [18]:
SYSTEM_PROMPT = (
    "You are a helpful assistant with access to simulated tools. "
    "Use the supplied tools when needed and clearly describe their results as simulated. "
    "Do not invent tool results. Ask for clarification if required arguments are missing. "
    "After receiving tool results, provide a concise answer in at most 100 words."
)


In [19]:
print(SYSTEM_PROMPT)

You are a helpful assistant with access to simulated tools. Use the supplied tools when needed and clearly describe their results as simulated. Do not invent tool results. Ask for clarification if required arguments are missing. After receiving tool results, provide a concise answer in at most 100 words.


## Functions

These functions simulate real API calls. **In production you would replace them with actual service calls.**

In [20]:
def get_weather(city: str, units: str = "celsius") -> dict:
    """Simulate a weather API response."""
    base_temp = {
        "celsius": random.randint(5, 30),
        "fahrenheit": random.randint(40, 86)
    }
    temp = base_temp.get(units, base_temp["celsius"])
    conditions = random.choice(
        ["Sunny", "Partly Cloudy", "Overcast", "Light Rain", "Thunderstorms"])

    forecast = []
    for i in range(1, 4):
        day = (datetime.now() + timedelta(days=i)).strftime("%A %b %d")
        forecast.append({
            "day":
            day,
            "high":
            temp + random.randint(-3, 5),
            "low":
            temp - random.randint(2, 8),
            "conditions":
            random.choice(["Sunny", "Cloudy", "Rain"])
        })

    return {
        "city": city,
        "temperature": temp,
        "units": units,
        "conditions": conditions,
        "humidity": random.randint(30, 90),
        "wind_kph": random.randint(5, 40),
        "forecast": forecast
    }


def get_stock_price(ticker: str) -> dict:
    """Simulate a stock API response."""
    prices = {
        "AAPL": 344.56,
        "MSFT": 521.30,
        "GOOGL": 278.90,
        "AMZN": 240.45,
        "NVDA": 204.20,
        "TSLA": 399.80
    }
    price = prices.get(ticker.upper(), round(random.uniform(20, 500), 2))
    change_pct = round(random.uniform(-4, 4), 2)

    return {
        "ticker": ticker.upper(),
        "price": price,
        "change_pct": change_pct,
        "change_direction": "up" if change_pct > 0 else "down",
        "52w_high": round(price * random.uniform(1.1, 1.4), 2),
        "52w_low": round(price * random.uniform(0.5, 0.85), 2),
        "volume": f"{random.randint(10, 120)}M",
        "market_cap": f"${random.randint(100, 3500)}B"
    }


def convert_currency(amount: float, from_currency: str,
                     to_currency: str) -> dict:
    """Simulate a currency conversion API."""
    rates = {
        ("USD", "EUR"): 0.92,
        ("EUR", "USD"): 1.09,
        ("USD", "GBP"): 0.79,
        ("GBP", "USD"): 1.27,
        ("USD", "JPY"): 149.50,
        ("JPY", "USD"): 0.0067,
        ("EUR", "GBP"): 0.86,
        ("GBP", "EUR"): 1.16,
        ("EUR", "JPY"): 162.80,
        ("USD", "CNY"): 7.24,
    }
    key = (from_currency.upper(), to_currency.upper())
    rate = rates.get(key, round(random.uniform(0.5, 150), 4))

    return {
        "from": from_currency.upper(),
        "to": to_currency.upper(),
        "amount": amount,
        "rate": rate,
        "converted": round(amount * rate, 2),
        "timestamp": datetime.now().isoformat()
    }


def search_web(query: str, max_results: int = 3) -> dict:
    """Simulate a web search API."""
    results = [{
        "title": f"Result {i+1} for '{query}'",
        "snippet":
        f"This is a relevant excerpt about {query} from source {i+1}...",
        "url": f"https://example.com/result{i+1}"
    } for i in range(min(max_results, 5))]
    return {
        "query": query,
        "total_results": random.randint(1000, 50000),
        "results": results
    }


def send_email(to: str, subject: str, body: str) -> dict:
    """Simulate sending an email."""
    return {
        "status": "sent",
        "to": to,
        "subject": subject,
        "message_id": f"msg-{random.randint(100000, 999999)}",
        "timestamp": datetime.now().isoformat()
    }


# Registry: maps tool name → Python callable
TOOL_REGISTRY = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
    "search_web": search_web,
    "send_email": send_email,
}

## Execute structured tool calls

The API returns function names and JSON argument strings in `message.tool_calls`.
Only registered Python functions can be executed. Errors are raised explicitly.
Each result is returned as a `tool` message with the matching `tool_call_id`.

In [21]:
def execute_tool_call(tool_call) -> dict:
    name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)
    if not isinstance(arguments, dict):
        raise ValueError(f"Arguments for {name!r} must be a JSON object.")
    if name not in TOOL_REGISTRY:
        raise ValueError(f"Unknown tool requested by model: {name!r}")
    result = TOOL_REGISTRY[name](**arguments)
    return {"tool": name, "arguments": arguments, "result": result}

## Full tool-use exchange

`chat` asks the model to select tools, executes every returned structured call,
then supplies the assistant message and matching `tool` results back to the model.
The final request uses `tool_choice="none"` to request a natural-language answer.

This preserves the original demo's two-stage workflow. With the tested CLI 0.12.1
and cached Qwen3 models, leaving tool choice on `auto` after tool execution caused
repeated tool calls, so finalization is explicit. Multiple calls in one response are
executed sequentially; this demo does not implement autonomous chained planning.


In [22]:
def chat(user_message: str, verbose: bool = True) -> str:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message},
    ]
    if verbose:
        print(f"USER: {user_message}")
    for tool_choice in ("auto", "none"):
        response = client.chat.completions.create(
            model=model_id,
            messages=messages,
            tools=TOOLS_OPENAI,
            tool_choice=tool_choice,
            temperature=0.0,
            max_tokens=2048,
        )
        if not response.choices:
            raise RuntimeError("The model returned no choices.")
        choice = response.choices[0]
        if choice.finish_reason == "length":
            raise RuntimeError("The model response was truncated; increase max_tokens.")
        message = choice.message
        if not message.tool_calls:
            if not message.content or not message.content.strip():
                raise RuntimeError("The model returned neither text nor tool calls.")
            if verbose:
                print(f"ASSISTANT: {message.content}")
            return message.content
        if tool_choice == "none":
            raise RuntimeError("The runtime returned tool calls despite tool_choice='none'.")
        messages.append(message.model_dump(exclude_none=True))
        for tool_call in message.tool_calls:
            result = execute_tool_call(tool_call)
            if verbose:
                print(f"TOOL: {result['tool']}({result['arguments']})")
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result["result"]),
            })
    raise RuntimeError("The model did not return a final answer after tool execution.")


## Examples

### Single tool call: Weather

In [23]:
_ = chat("What's the weather like in Paris right now? Use celsius.")

USER: What's the weather like in Paris right now? Use celsius.
TOOL: get_weather({'city': 'Paris', 'units': 'celsius'})
ASSISTANT: 
The current weather in Paris is as follows:

- **Temperature**: 19°C  
- **Conditions**: Thunderstorms  
- **Humidity**: 70%  
- **Wind Speed**: 5 km/h  

So, it's currently very hot with thunderstorms in Paris.


### Single tool call: Stock price
This is a simulated example.

In [24]:
_ = chat("How is NVDA stock doing today? Give me the price and 52-week range.")

USER: How is NVDA stock doing today? Give me the price and 52-week range.
TOOL: get_stock_price({'ticker': 'NVDA'})
ASSISTANT: 
The NVDA stock is currently trading at $204.2. The 52-week high for NVDA is $235.42, and the 52-week low is $119.42. The volume is 16 million, and the market cap is $906 billion.


> Go to the next notebook